   
Videos are about 2min long\
We have 5 videos for each emotion

Goal: \
#1 Clean ECG data (e.g., remove artefactual areas, ectopic heart beats, etc.) and identify R-R intervals\
    

In [2]:

# libraries

import json
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import zscore
from scipy.ndimage import gaussian_filter1d
from scipy.interpolate import interp1d

Prepare lists to use later

In [3]:

# targets
targets = ['P03_joy', 'P29_joy', 'P34_joy', 'P42_joy', 'P44_joy',
           'P12_sadness', 'P15_sadness', 'P27_sadness', 'P34_sadness', 'P44_sadness', 
           'P03_anger', 'P15_anger', 'P22_anger', 'P38_anger', 'P42_anger',
           'P17_pride', 'P22_pride', 'P29_pride', 'P42_pride', 'P44_pride',
           'P03_shame', 'P22_shame', 'P37_shame', 'P38_shame', 'P45_shame', 
           'P12_neutral', 'P17_neutral', 'P29_neutral', 'P34_neutral', 'P38_neutral']

# perceivers
perceivers = ["S01", "S02", "S03", "S04", "S05", "S06", "S07", "S08", "S11", "S15", "S18", "S20", "S23",
              "S24", "S28", "S34", "S37", "S42", "S44", "S45", "S47", "S49", "S51", "S54", "S55", "S57", "S64", "S65"]

# data folder
data_folder = r"physiological_data"


Functions to clean the data -> exclude 'invalid areas' (artefactual) and calculate R-R intervals

Also smooth the data with a 10s sliding window (both 'average' and 'gaussian' methods)

In [ ]:
# helper functions

# determines if two intervals overlap (from biotop) => this will help to remove invalid areas (e.g., noisy signal) from data (later)
def does_overlap(intv1, intv2):
    (a1, b1) = intv1
    if a1 > b1: (a1, b1) = (b1, a1)
    (a2, b2) = intv2
    if a2 > b2: (a2, b2) = (b2, a2)
    overlapmin = max([a1, a2])
    overlapmax = min([b1, b2])
    return overlapmin <= overlapmax # true if overlap

# filter out any interval overlapping with invalid area
def exclude_invalid(peak_times, invalid_areas):
    valid_intervals = []
    for i in range(len(peak_times) - 1):  # -1 so it does not go out of range
        start_time = peak_times[i]  # peak 1
        end_time = peak_times[i + 1]  # peak 2
        interval = (start_time, end_time)
        if any(does_overlap(interval, invalid_area) for invalid_area in invalid_areas):
            valid_intervals.append((start_time, np.nan)) # marking the end as NaN to signal not to use it
        else:
            valid_intervals.append(interval)  # if not overlap with invalid area, add to list
    return valid_intervals

# define peak times and RR intervals
def calculate_valid_RRs(ecg_data, label):
    peak_times = np.array([data['t'] for data in ecg_data['peaks'][label]]) # identify peak times
    peak_times.sort() # sort in order
    invalid_areas = [(data['start'], data['end']) for data in ecg_data['invalid'][label]] # identify invalid areas
    valid_intervals = exclude_invalid(peak_times, invalid_areas) # exclude invalid
    time_diffs = [end - start if ecg_data['peaks'][label][i]['valid'] and
                  ecg_data['peaks'][label][i-1]['valid'] else None
                  for i, (start, end) in enumerate(valid_intervals)] # identify RR intervals
    peak_times = peak_times[:min(len(peak_times[:-1]), len(time_diffs))]   # peaks - 1 so n peaks = n intervals (otherwise further code do not work properly; maybe there is a better solution?)
    time_diffs = np.array([np.nan if x is None else x for x in time_diffs]) # convert 'none' to 'nan' (useful for correlations later)
    return peak_times, time_diffs # return list of peaks (peak_times) and RR intervals (time_diffs)

# load ECG data from JSON
def load_ecg_data(file_path):
    try:
        with open(file_path, 'r') as file:
            return json.load(file)
    except FileNotFoundError:
        return {} # just in case

# clean perceiver ECG data => separate from target cleaning, because different structure (file name, channel name, etc.)
def clean_perceiver_ecg(data_folder, perceiver, target):
    perceiver_file = os.path.join(data_folder, f"{perceiver}_{target}_peaks.json") # open perceiver file
    if not os.path.exists(perceiver_file):
        return [], [] # (perceivers did not watch all targets) 
    ecg_data_perceiver = load_ecg_data(perceiver_file)
    peak_times_perceiver, time_diffs_perceiver = calculate_valid_RRs(ecg_data_perceiver, "ECG, X, RSPEC-R") # calculate RR intervals
    return peak_times_perceiver, time_diffs_perceiver

# clean target ECG data => ** I should combine this with clean_perceiver_ecg at some point
def clean_target_ecg(data_folder, target):
    target_file = os.path.join(data_folder, f"{target}_peaks.json") # open target file
    ecg_data_target = load_ecg_data(target_file)
    target_id = target.split('_')[0] # take just target number (before the '_'), not whole video name
    peak_times_target, time_diffs_target = calculate_valid_RRs(ecg_data_target, f"ECG {target_id}") # calculate RR intervals
    return peak_times_target, time_diffs_target


Visualize data, and at the same time explore different strategies of ectopic beat detection. 


In [ ]:


count = 0


for target in targets:
    for perceiver in perceivers:
        
        
        if count >= 20:
            break # I added this to not produce all plots (28 perceivers * 12 targets), only 20 first ones for verification
        
        
        ecg_data_perceiver = load_ecg_data(os.path.join(data_folder, f"{perceiver}_{target}_peaks.json")) 
        perceiver_peak_times, perceiver_time_diffs = clean_perceiver_ecg(data_folder, perceiver, target) # extract peak times and RR intervals
        
        if len(perceiver_time_diffs) == 0:
            continue # skip if no data (perceivers did not watch all targets)
        
        print(f"RR-intervals: {perceiver_time_diffs}") # to check anomalies (just in case)
        
        invalid_areas = ecg_data_perceiver.get("invalid", {}).get("ECG, X, RSPEC-R", []) # identify invalid areas to mark them in the plot

        
        
        
        # malik rule: RR-intervals differing by more than the 20 % of the next RR interval are removed.
        outlier_malik = [RR for RR in range(len(perceiver_time_diffs) - 1) if abs(perceiver_time_diffs[RR] - perceiver_time_diffs[RR + 1]) > 0.2 * perceiver_time_diffs[RR]] # -1 otherwise out of bounds?
        
        # kamath rule: R-R intervals increasing or decreasing by more than 32.5% or by more than 24.5% from the previous interval are removed.
        outlier_kamath = [RR for RR in range(len(perceiver_time_diffs) - 1) if 0 < perceiver_time_diffs[RR + 1] - perceiver_time_diffs[RR] > .325 * perceiver_time_diffs[RR] or 
                                                                                0 < (perceiver_time_diffs[RR] - perceiver_time_diffs[RR+1] > .245 * perceiver_time_diffs[RR])]
                 
        # karlsson rule: RR-intervals differing by more than the 20 % of the mean of previous and next RR-interval are removed.
        outlier_karlsson = [RR for RR in range(1, len(perceiver_time_diffs) - 1) if abs(((perceiver_time_diffs[RR - 1] + perceiver_time_diffs[RR + 1]) / 2) - perceiver_time_diffs[RR]) > 0.2 * ((perceiver_time_diffs[RR - 1] + perceiver_time_diffs[RR + 1]) / 2)]
        
        # acar rule: RR-intervals differing by more than the 20 % of the mean of last 9 RR intervals are removed.
        outlier_acar = [RR for RR in range(9, len(perceiver_time_diffs)) if abs(perceiver_time_diffs[RR] - np.mean(perceiver_time_diffs[RR-9:RR])) > 0.2 * np.mean(perceiver_time_diffs[RR-9:RR])]

        # sd rule: RR-intervals exceeding 3 SD from the mean of all RR-intervals are removed. 
        outlier_sd = [i for i in range(len(perceiver_time_diffs) -1) if abs(zscore(perceiver_time_diffs)[i]) > 3]
    
    
    
        plt.figure(figsize=(12, 4)) # create plot

        plt.plot(perceiver_peak_times, perceiver_time_diffs, color='black', linestyle='-', marker='o', markersize=3) # raw RR intervals
        for invalid in invalid_areas: # highlight invalid areas in ECG
            plt.axvspan(invalid["start"], invalid["end"], color="grey", alpha=0.3)
            
        for i in outlier_malik: # identify ectopic beats according to each rule (different color and shape)
            plt.scatter(perceiver_peak_times[i], perceiver_time_diffs[i], color='royalblue', s=200, marker='o')            
        for i in outlier_kamath:
            plt.scatter(perceiver_peak_times[i], perceiver_time_diffs[i], color='green', s=150, marker='^')
        for i in outlier_karlsson:
            plt.scatter(perceiver_peak_times[i], perceiver_time_diffs[i], color='red', s=100, marker='s')
        for i in outlier_acar:
            plt.scatter(perceiver_peak_times[i], perceiver_time_diffs[i], color='magenta', s=70, marker='D')
        for i in outlier_sd:
            plt.scatter(perceiver_peak_times[i], perceiver_time_diffs[i], color='orange', s=50, marker='X')
            
            
            
        plt.xlabel('time (s)')
        plt.ylabel('RR intervals')
        plt.title(f'RR intervals: {perceiver} evaluating {target}')

        plt.tight_layout()
        plt.show()
        
        
        
        count += 1
    if count >= 20:
        break        

Now remove ectopic beats based on Karlsson method: RR-intervals differing by more than the 20 % of the mean of previous and next RR-interval are removed.

Then save the R-R intervals in a new file.

In [6]:

# function to identify ectopic beats
def identify_ectopic(time_diffs):
    ectopic_beats = [RR for RR in range(1, len(time_diffs) - 1)
                     if abs(((time_diffs[RR - 1] + time_diffs[RR + 1]) / 2) - time_diffs[RR]) 
                     > 0.2 * ((time_diffs[RR - 1] + time_diffs[RR + 1]) / 2)]       
        
    return ectopic_beats # beats over 20% mean of last and next RR interval
        
cleaned_folder = "physiological_data/ECG_cleaned_data" # new folder to put in the cleaned data

# loop through all data to clean it
for target in targets:
    
    target_peak_times, target_time_diffs = clean_target_ecg(data_folder, target) # identify peak times and RR intervals
    target_ectopic = identify_ectopic(target_time_diffs) # get the ectopic beats
    target_time_diffs[target_ectopic] = np.nan # mark ectopic beats as NaN
    
    # save cleaned data   
    cleaned_data_target = {"peak_times": target_peak_times.tolist(), # first, save a list of peak times
                           "RR_intervals": target_time_diffs.tolist(), # second, save a list of RR intervals (ectopic beats as NaN)
                           "ectopic": len(target_ectopic)} # last, track nb of ectopic beats
    
    target_output_file = f"{cleaned_folder}/{target}_RR.json"
    with open(target_output_file, 'w') as file: # w = write
        json.dump(cleaned_data_target, file) # put data in .json file
               
    for perceiver in perceivers: # now do the same for perceivers
        
        perceiver_peak_times, perceiver_time_diffs = clean_perceiver_ecg(data_folder, perceiver, target) # clean perceiver ECG data
        
        if len(perceiver_time_diffs) == 0:
            continue # skip if no data
        
        perceiver_ectopic = identify_ectopic(perceiver_time_diffs)
        perceiver_time_diffs[perceiver_ectopic] = np.nan # mark ectopic beats as NaN
       
        cleaned_data_perceiver = {
            "peak_times": perceiver_peak_times.tolist(),
            "RR_intervals": perceiver_time_diffs.tolist(),
            "ectopic": len(perceiver_ectopic)}
        
        perceiver_output_file = f"{cleaned_folder}/{perceiver}_{target}_RR.json"
            
        with open(perceiver_output_file, 'w') as file: # w = write mode
            json.dump(cleaned_data_perceiver, file) # puts data in a new .json file
                                                            